# 03. 기상 데이터 전처리

**읽는 것**: `data/external/OBS_ASOS_TIM_*.csv` (기상청 ASOS 시간자료, cp949)

**만드는 것**: `data/processed/weather_daily.csv`, `weather_timegb.csv`

원본 파일은 읽기만 하고 절대 수정하지 않습니다.

# 1.경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
RAW_CARD = BASE / "data/raw/card"
EXT = BASE / "data/external"
OUT = BASE / "data/processed"
DOCS = BASE / "docs"
OUT.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

## 2. 읽기 + 컬럼명 변경

| 원본 | 변경 |
|---|---|
| 지점 | stn |
| 지점명 | stn_name |
| 일시 | dt |
| 기온(°C) | temp |
| 강수량(mm) | rain |
| 풍속(m/s) | wind |
| 습도(%) | rh |

`region`은 카드 데이터와 같은 값(강남, 춘천)으로 맞춥니다. 서울(108) 관측소는 종로구에 있음.

In [ ]:
w = pd.read_csv(next(EXT.glob("OBS_ASOS_TIM_*.csv")), encoding="cp949")
w = w.rename(columns={"지점": "stn", "지점명": "stn_name", "일시": "dt",
                      "기온(°C)": "temp", "강수량(mm)": "rain",
                      "풍속(m/s)": "wind", "습도(%)": "rh"})
w["region"] = w["stn"].map({108: "강남", 101: "춘천"})
print(w.shape)                       # (8832, 8) 예상
print(w["region"].value_counts())    # 각 4416 예상

## 3. 형 변환 + 결측 처리


In [ ]:
w["dt"] = pd.to_datetime(w["dt"])
w["rain"] = w["rain"].fillna(0)
w["temp"] = w.groupby("stn")["temp"].transform(lambda s: s.interpolate())
print(w.isna().sum().sum())          # 0 예상

## 4. 파생 변수

- `feels`: 기상청 여름철 체감온도 (기온 + 습도)
- `time_gb`: 카드 데이터 시간대와 같은 구간

In [ ]:
T, RH = w["temp"], w["rh"]
Tw = (T * np.arctan(0.151977 * (RH + 8.313659) ** 0.5) + np.arctan(T + RH)
      - np.arctan(RH - 1.67633) + 0.00391838 * RH ** 1.5 * np.arctan(0.023101 * RH)
      - 4.686035)
w["feels"] = (-0.2442 + 0.55399 * Tw + 0.45535 * T - 0.0022 * Tw ** 2
              + 0.00278 * Tw * T + 3.0)

w["date"] = w["dt"].dt.normalize()
w["time_gb"] = pd.cut(w["dt"].dt.hour, [-1, 5, 11, 17, 23],
                      labels=["00_05", "06_11", "12_17", "18_23"]).astype(str)

## 5. 일 단위 집계

- `heat`: 일최고기온 33℃ 이상
- `heat_streak`: 연속 폭염 일수
- `tmax_lag1`: 전날 최고기온

In [ ]:
daily = w.groupby(["region", "date"]).agg(
    tmax=("temp", "max"), tmean=("temp", "mean"), tmin=("temp", "min"),
    feels_max=("feels", "max"), rh_mean=("rh", "mean"),
    rain=("rain", "sum"), wind_mean=("wind", "mean")).reset_index()

daily["heat"] = (daily["tmax"] >= 33).astype(int)
daily["heat_streak"] = (daily.groupby(["region", (daily["heat"] == 0).cumsum()])["heat"]
                        .cumsum())
daily["tmax_lag1"] = daily.groupby("region")["tmax"].shift(1)
print(daily.groupby("region")["heat"].sum())   # 강남 24, 춘천 24 예상

## 6. 시간대 단위 집계

카드 `panel_time`과 맞추기 위해 00_05 제외. 일 단위 변수와 겹치지 않게 `_tg`를 붙임.

In [ ]:
by_time = (w[w["time_gb"] != "00_05"]
           .groupby(["region", "date", "time_gb"])
           .agg(temp_tg=("temp", "mean"), tmax_tg=("temp", "max"),
                feels_max_tg=("feels", "max"), rain_tg=("rain", "sum"))
           .reset_index())

## 7. 저장

In [ ]:
daily.to_csv(OUT / "weather_daily.csv", index=False, encoding="utf-8-sig")
by_time.to_csv(OUT / "weather_timegb.csv", index=False, encoding="utf-8-sig")
print(daily.shape, by_time.shape)    # (368, 12) (1104, 7) 예상